# Insulin Resistance Prediction UI

A simple Gradio interface for testing the trained logistic regression model.

In [1]:
%pip install gradio pandas joblib

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 23.2.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
from pathlib import Path

import gradio as gr
import pandas as pd
from joblib import load

MODEL_PATH = Path("models") / "research_grade_type2_logistic_regression_model.pkl"

FEATURE_COLUMNS = [
    "age",
    "gender_encoded",
    "BMI",
    "fasting_glucose",
    "HbA1c",
    "insulin",
    "triglycerides",
    "blood_pressure",
    "physical_activity_encoded",
    "cholesterol_ratio",
]

GENDER_MAP = {"Female": 0, "Male": 1}
ACTIVITY_MAP = {"Low": 0, "Medium": 1, "High": 2}

model = load(MODEL_PATH)

model_features = list(getattr(model, "feature_names_in_", FEATURE_COLUMNS))
if model_features != FEATURE_COLUMNS:
    raise ValueError(f"Unexpected model features: {model_features}")

c:\Users\A7maD Elsa3danY\AppData\Local\Programs\Python\Python311\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
def predict_insulin_resistance(
    age,
    gender,
    bmi,
    fasting_glucose,
    hba1c,
    insulin,
    triglycerides,
    hdl,
    ldl,
    blood_pressure,
    physical_activity,
):
    if hdl <= 0:
        raise gr.Error("HDL must be greater than 0 to calculate the cholesterol ratio.")

    cholesterol_ratio = ldl / hdl
    sample = pd.DataFrame(
        [[
            age,
            GENDER_MAP[gender],
            bmi,
            fasting_glucose,
            hba1c,
            insulin,
            triglycerides,
            blood_pressure,
            ACTIVITY_MAP[physical_activity],
            cholesterol_ratio,
        ]],
        columns=FEATURE_COLUMNS,
    )

    probability = float(model.predict_proba(sample)[0][1])
    prediction = int(probability >= 0.5)
    label = "Insulin resistance likely" if prediction else "Insulin resistance unlikely"
    confidence = probability if prediction else 1 - probability

    summary = (
        f"### {label}\n"
        f"Risk probability: **{probability:.1%}**\n\n"
        f"Model confidence for this label: **{confidence:.1%}**\n\n"
        f"Calculated cholesterol ratio (LDL / HDL): **{cholesterol_ratio:.2f}**"
    )

    probabilities = {
        "No insulin resistance": 1 - probability,
        "Insulin resistance": probability,
    }

    return summary, probabilities, sample


example_healthy = [32, "Female", 22.5, 88, 4.9, 9.0, 165, 48, 105, 118, "High"]
example_at_risk = [62, "Male", 34.0, 155, 7.1, 35.0, 245, 32, 150, 150, "Low"]

In [4]:
theme = gr.themes.Soft(
    primary_hue="teal",
    secondary_hue="slate",
    neutral_hue="zinc",
).set(
    body_background_fill="#f7faf9",
    block_background_fill="#ffffff",
    button_primary_background_fill="#0f766e",
    button_primary_background_fill_hover="#115e59",
)

css = """
.gradio-container {max-width: 1120px !important; margin: auto !important;}
#title {text-align: center; margin-bottom: 10px;}
#title h1 {color: #134e4a; font-weight: 760;}
#title p {color: #475569; font-size: 16px;}
.panel-note {color: #64748b; font-size: 13px;}
"""

with gr.Blocks(theme=theme, css=css, title="Insulin Resistance Predictor") as demo:
    gr.Markdown(
        """
        # Insulin Resistance Predictor
        Enter patient measurements to estimate the probability of insulin resistance using the trained logistic regression model.
        """,
        elem_id="title",
    )

    with gr.Row():
        with gr.Column(scale=1):
            gr.Markdown("**Patient profile**")
            age = gr.Slider(19, 90, value=54, step=1, label="Age")
            gender = gr.Radio(["Female", "Male"], value="Male", label="Gender")
            bmi = gr.Slider(10, 55, value=26.0, step=0.1, label="BMI")
            physical_activity = gr.Radio(["Low", "Medium", "High"], value="Medium", label="Physical activity")

            gr.Markdown("**Blood markers**")
            fasting_glucose = gr.Slider(60, 260, value=105, step=0.1, label="Fasting glucose")
            hba1c = gr.Slider(2.0, 12.5, value=5.5, step=0.1, label="HbA1c")
            insulin = gr.Slider(2, 45, value=24, step=0.1, label="Insulin")

        with gr.Column(scale=1):
            gr.Markdown("**Cardiometabolic markers**")
            triglycerides = gr.Slider(80, 350, value=206, step=0.1, label="Triglycerides")
            hdl = gr.Slider(10, 80, value=36, step=0.1, label="HDL cholesterol")
            ldl = gr.Slider(50, 230, value=126, step=0.1, label="LDL cholesterol")
            blood_pressure = gr.Slider(80, 210, value=136, step=0.1, label="Blood pressure")

            predict_button = gr.Button("Predict", variant="primary")
            result = gr.Markdown(label="Prediction")
            probability_output = gr.Label(label="Class probabilities")

    model_input = gr.Dataframe(label="Model input row", interactive=False, wrap=True)

    inputs = [
        age,
        gender,
        bmi,
        fasting_glucose,
        hba1c,
        insulin,
        triglycerides,
        hdl,
        ldl,
        blood_pressure,
        physical_activity,
    ]

    predict_button.click(
        predict_insulin_resistance,
        inputs=inputs,
        outputs=[result, probability_output, model_input],
    )

    gr.Examples(
        examples=[example_healthy, example_at_risk],
        inputs=inputs,
        outputs=[result, probability_output, model_input],
        fn=predict_insulin_resistance,
        cache_examples=False,
    )

demo.launch()

C:\Users\A7maD Elsa3danY\AppData\Local\Temp\ipykernel_9052\2037597390.py:20: UserWarning: The parameters have been moved from the Blocks constructor to the launch() method in Gradio 6.0: theme, css. Please pass these parameters to launch() instead.
  with gr.Blocks(theme=theme, css=css, title="Insulin Resistance Predictor") as demo:


* Running on local URL:  http://127.0.0.1:7860
* To create a public link, set `share=True` in `launch()`.
